# 07 — Discovering STAC catalogs from browser-native notebooks

This lab connects four public STAC Browsers to their machine-readable catalog APIs: [UN Biodiversity Lab](https://stac.unbiodiversitylab.org/?.language=en), [Copernicus Data Space](https://browser.stac.dataspace.copernicus.eu/), [USGS LandsatLook](https://landsatlook.usgs.gov/stac-browser/?.language=en), and [WorldPop](https://stac.worldpop.org/?.language=en).

A STAC Browser is a human-friendly client. The STAC API behind it is what a notebook can query reproducibly.

In [ ]:
import sys
if sys.platform == 'emscripten':
    import micropip
    await micropip.install(['geolibre==3.0.0', 'pyodide-http'])
    import pyodide_http
    pyodide_http.patch_all()
from geolibre_lite import LiteMap as Map
import requests
from IPython.display import display, Markdown


In [ ]:
CATALOGS = {
    'UN Biodiversity Lab': {
        'api': 'https://unbl-prod-stac.azurewebsites.net/',
        'browser': 'https://stac.unbiodiversitylab.org/?.language=en',
    },
    'Copernicus Data Space': {
        'api': 'https://stac.dataspace.copernicus.eu/v1/',
        'browser': 'https://browser.stac.dataspace.copernicus.eu/',
    },
    'USGS LandsatLook': {
        'api': 'https://landsatlook.usgs.gov/stac-server/',
        'browser': 'https://landsatlook.usgs.gov/stac-browser/?.language=en',
    },
    'WorldPop': {
        'api': 'https://api.stac.worldpop.org',
        'browser': 'https://stac.worldpop.org/?.language=en',
    },
}

def get_json(url, **kwargs):
    response = requests.get(url, timeout=60, **kwargs)
    response.raise_for_status()
    return response.json()

catalog_rows = []
catalog_roots = {}
for name, info in CATALOGS.items():
    try:
        root = get_json(info['api'].rstrip('/') + '/')
        catalog_roots[name] = root
        catalog_rows.append({
            'catalog': name,
            'title': root.get('title'),
            'stac_version': root.get('stac_version'),
            'api_root': info['api'],
            'browser': info['browser'],
            'status': 'API reachable from this browser',
        })
    except Exception as exc:
        # A STAC Browser can work even when its API blocks cross-origin XHR.
        catalog_rows.append({
            'catalog': name,
            'title': 'Unavailable to in-browser requests',
            'stac_version': None,
            'api_root': info['api'],
            'browser': info['browser'],
            'status': f'Browser CORS/authentication restriction: {type(exc).__name__}',
        })

import pandas as pd
pd.DataFrame(catalog_rows)


## Open the human-facing browsers

Use these links to inspect the same catalogs visually, then use the API URLs above for reproducible code. The UN Biodiversity Lab catalog may return metadata while withholding protected asset URLs unless you authenticate in its browser.

In [ ]:
display(Markdown('\n'.join(f"- [{name}]({info['browser']}) — API: `{info['api']}`" for name, info in CATALOGS.items())))


In [ ]:
# The catalog's links advertise the standard endpoints a client can follow.
for name, root in catalog_roots.items():
    rels = sorted({link.get('rel') for link in root.get('links', []) if link.get('rel')})
    print(name, '=>', ', '.join(rels))
missing = sorted(set(CATALOGS) - set(catalog_roots))
if missing:
    print('Skipped from direct API introspection because the browser blocked the request:', ', '.join(missing))


## GeoLibre context map

The map is not downloading imagery here. It gives the four catalogs a geographic context; the next notebook turns STAC search results into GeoAI-ready features.

In [ ]:
m = Map(center=(-20, 20), zoom=1.5, height='560px')
m.add_marker(-74.0, 40.7, name='USGS / LandsatLook', properties={'browser': CATALOGS['USGS LandsatLook']['browser']})
m.add_marker(12.5, 42.0, name='Copernicus Data Space', properties={'browser': CATALOGS['Copernicus Data Space']['browser']})
m.add_marker(0, 0, name='UN Biodiversity Lab', properties={'browser': CATALOGS['UN Biodiversity Lab']['browser']})
m.add_marker(104.9, 12.6, name='WorldPop', properties={'browser': CATALOGS['WorldPop']['browser']})
m


## Data & software citations

- [STAC specification](https://stacspec.org/) and [STAC API specification](https://github.com/radiantearth/stac-api-spec).
- [UN Biodiversity Lab STAC Browser](https://stac.unbiodiversitylab.org/?.language=en); public API root: `https://unbl-prod-stac.azurewebsites.net/`.
- [Copernicus Data Space STAC Browser](https://browser.stac.dataspace.copernicus.eu/); public API root: `https://stac.dataspace.copernicus.eu/v1/`.
- [USGS LandsatLook STAC Browser](https://landsatlook.usgs.gov/stac-browser/?.language=en); public API root: `https://landsatlook.usgs.gov/stac-server/`.
- [WorldPop STAC Browser](https://stac.worldpop.org/?.language=en); public API root: `https://api.stac.worldpop.org`.
- [GeoLibre](https://geolibre.app/) and [GeoAI](https://opengeoai.org/).